### About task

In this assignment you:
- get familiar with what happens "inside" support vector machine and logistic regression
- get familiar with probability calibration
- learn variable transformation methods and feature selection methods
- try to evaluate the economic effect of the model

#### Deadline:

- Soft deadline 1-Mart-2024
- Hard deadline 8-Mart-2024


# Part 1. SVM, LR and probability calibration (2 points + 0.5 bonus)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

#### Task 1.1. Compare the methods (0.5 points)


Let's generate synthetic data.

In [2]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=10000, n_features=10, n_informative=5, n_redundant=5,
    random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

__Random classifier__

First, let's define the simplest model, which produces a random answer for each object. Using the test sample, we calculate AUC-ROC, AUC-PR.

In [3]:
from sklearn.dummy import DummyClassifier
random_classifier = DummyClassifier(strategy='uniform', random_state=42).fit(X_train, y_train)
y_random = random_classifier.predict_proba(X_test)[:,1]
y_random

NameError: name 'X_train' is not defined

**Question:** We are solving a binary classification problem, but y\_random contains some fractional numbers, not 0/1. Why?

**Answer**: #your answer here

*Below is an **example** of working* with the built-in `sklearn` functions for drawing ROC and PR curves and saving metrics. The pipeline can be changed as you wish.

In [ ]:
from sklearn.metrics import auc
from sklearn.metrics import average_precision_score
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import PrecisionRecallDisplay

from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay

In [ ]:
def depict_pr_roc(y_true, y_pred, classifier_name='Some Classifier', ax=None):
  if ax is None:
    fig, ax = plt.subplots(1, 2, figsize=(11, 5))

  print(classifier_name, 'metrics')
  PrecisionRecallDisplay.from_predictions(y_true, y_pred, ax=ax[0], name=classifier_name)
  precision, recall, _ = precision_recall_curve(y_true, y_pred)
  print('AUC-PR: %.5f' % auc(recall, precision))
  ax[0].set_title("PRC")
  ax[0].set_ylim(0, 1.1)

  RocCurveDisplay.from_predictions(y_true, y_pred, ax=ax[1], name=classifier_name)
  print('AUC-ROC: %.5f' % roc_auc_score(y_true, y_pred))
  ax[1].set_title("ROC")
  ax[1].set_ylim(0, 1.1)

  plt.tight_layout()
  plt.legend()


depict_pr_roc(y_test, y_random, 'Random Classifier')

In [ ]:
# dataframe for comparing
# classifiers for different metrics
df_metrics = pd.DataFrame(columns=['AUC_PR', 'AUC_ROC', 'reg_const'])
precision, recall, _ = precision_recall_curve(y_test, y_random)
  # adding another line with method metrics
df_metrics.loc['Random Classifier'] = [
      auc(recall, precision),
      roc_auc_score(y_test, y_random),
      0
]

# Likewise the results of the following experiments can be collected in a table
df_metrics

__Support Vector Machine (Linear Kernel)__


Train support vector machines. In the classroom, we went through the linear version without a kernel transition, which corresponds to the linear kernel (`LinearSVC` or `SVC(kernel='linear')` in `sklearn.svm`).

Choose the regularization parameter `C` (you can use cross-validation or separate the validation sample from the training one).

In [ ]:
#your code here

On the test part, 
* calculate AUC-ROC, AUC-PR with an accuracy of 5 digits. 
* Plot ROC and PR curves



In [4]:
#your code here

Compare them with the responses of a random classifier. 

Note that with `sklearn` you can calculate both metrics in two different ways - what are their differences? Compare results.

In [ ]:
#your code here

__Logistic Regression__

A similar task for logistic regression with L2 regularizer:


* fit hyperparameter C using AUC-PR metric
* draw ROC, PR curves for the test part
* output metrics for test data and compare them with the results of a random classifier

In [5]:
#your code here

Draw ROC, PR curves for test part for all 3 classifiers on one graph

In [ ]:
#your code here

**Question:** Compare the results of LR and SVM in terms of all calculated quality criteria, explain the differences (if any).

**Answer:** # your answer here

#### __Task 1.2. Vizualization SVM, LR__ (0.5 point)

There are some "support vectors" in the name of the support vector machine. 

* Generate a synthetic dataset using `make_classification` with 2 features, train support vector machines on it. This time, if necessary, use `SVC(kernel='linear')` instead of `LinearSVC`, as it is the only one with support vector information.
* Visualize the dividing line, all objects, and select the support vectors (the `support_vectors_` attribute). 

In [6]:
#your code here

In [ ]:
def plot_svm_2D(X, y, model,  plot_support=True):

    #создали сетку
    xx = np.linspace(X[:,0].min(), X[:,0].max(), 30)
    yy = np.linspace(X[:,1].min(), X[:,1].max(), 30)
    YY, XX = np.meshgrid(yy, xx)
    xy = np.vstack([XX.ravel(), YY.ravel()]).T

    #Ответы модели для сетки для отрисовки разделяющей прямой
    Z = #your code here

    plt.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1], alpha=0.5, linestyles=['--', '-', '--'])


    #Отрисовали выборку
    plt.scatter(#your code here
                )

    #Отрисовали опорные векторы
    if plot_support:
        plt.scatter(#your code here
                   label='support vectors',
                   s=100,
                   linewidth=1,
                   edgecolor="blue",
                   facecolors='none')

    plt.legend()

plot_svm_2D(X, y, model)

**Question:** what objects are identified as “support” vectors?

**Answer:** #your answer here

Unlike the support vector machine, logistic regression does not try to build a separating hyperplane, but approximates the likelihood of positive answers $p(y=+1|x)$ at each point in the object space. Try to draw this distribution on a plane, do not forget to mark all the objects on it. The `plt.pcolor` function can help you.

In [7]:
#your code here

In [ ]:
def plot_logreg_2D(X, y, model):

    #создали сетку
    xx = np.linspace(X[:,0].min(), X[:,0].max(), 100)
    yy = np.linspace(X[:,1].min(), X[:,1].max(), 100)
    YY, XX = np.meshgrid(yy, xx)
    xy = np.vstack([XX.ravel(), YY.ravel()]).T

    #Ответы модели для сетки для отрисовки распределения
    Z = #your code here
    Z = Z.reshape((xx.shape[0], -1)).T


    image = plt.imshow(Z, interpolation='nearest',
                       extent=(xx.min(), xx.max(), yy.min(), yy.max()),
                       aspect='auto', origin='lower', cmap=plt.cm.PuOr_r)

    #Отрисовали выборку
    plt.scatter(#your code here
                cmap=plt.cm.Paired)

    plt.colorbar(image)


plot_logreg_2D(X, y, model)

**Question:** How does the picture visualize the area where the model is unsure ($p(y=+1|x) = 0.5$)? How can this be justified theoretically?

**Answer:** # your answer here

__Task 2.__ Calibration of probabilities (1 point)

Let's move on to assessing the quality of the probabilities generated by the algorithms. Let's start with calibration curves.

Let's say the algorithm returns some numbers from zero to one. Are they good at assessing probability? 


Well-calibrated classifier should produce values such that among the samples for which it produces a value close to $\alpha$, approximately $\alpha \times$100%  actually belonged to the positive class. (For example, if the classifier returns 0.3 for some, then 30% of them should belong to class 1)

To do this, we divide the segment $[0, 1]$ into several small segments of the same length.

Let's consider the $i$-th segment with boundaries $[a_i, b_i]$ and the predictions $p_1, p_2, \dots, p_k$ that fall into it. Let the true answers $y_1, y_2, \dots, y_k$ correspond to them. If the algorithm produces the correct probabilities, then there should be approximately $(a_i + b_i) / 2$ units among these true answers. In other words, if you draw a curve in which the centers of the segments are plotted along the X axis, and the proportions of these single answers in the segments along the Y axis, then it should turn out to be diagonal. Below is a function that should draw such curves. There are two mistakes in it - find and correct them.

In [8]:
def plot_calibration_curve(y_test, preds):
    bin_middle_points = []
    bin_real_ratios = []
    n_bins = 10
    for i in range(n_bins):
        l = 1.0 / n_bins * i
        r = 1.0 / n_bins * (i + 1)
        bin_middle_points.append((l - r) / 2) 
        bin_real_ratios.append(np.min(y_test[(preds >= l) & (preds < r)] == 1)) 
    plt.figure(figsize=(6,6))
    plt.plot(bin_middle_points, bin_real_ratios)
    plt.ylim([-0.05, 1.05])
    plt.grid()

Generate synthetic data similar to those used in the very first task. Plot calibration curves on the test part for logistic regression and support vector machine (remember to convert its predictions to $[0;1]$).

Draw the calibration curve of a perfectly calibrated model (diagonal)

In [9]:
#your code here

**Question**: Are the curves for SVM, logistic regression well calibrated?

**Answer:** #your answer here

Examine the distribution of classifier responses (construct histograms using `plt.hist`).

In [6]:
#your code here

**Question:** How are they different? How can you explain this?

**Answer:** #your answer here

Use the `CalibratedClassifierCV` from `sklearn` to calibrate the probabilities of the support vector machine on training and use it to make predictions for the test set. Draw a calibration curve for them. 


In [10]:
#your code here

**Question:** Has the calibration curve improved?

**Answer:** #your answer here

__Bonus task: Author's solution__ (0.5 points)

Implement your function to calibrate the probabilities. Describe your approach and show the results. Keywords for inspiration: `Platt`, `Isotonic`.

In [11]:
#your code here

# Part 2. Working with variables (4 points + 1.5 bonus)

In this part, we will learn how to handle categorical variables, since it is not enough to encode them as numbers (this sets some order, which may not be on categorical variables). There are two main ways to handle categorical values:
- One-hot coding
- Counters (CTR, mean-target coding, ...) — each categorical feature is replaced by the average value of the target variable for all objects that have the same value in this feature.

Let's start with one-hot coding. Suppose our categorical feature $f_j(x)$ takes values ​​from the set $C=\{c_1, \dots, c_m\}$. Let's replace it with $m$ binary features $b_1(x), \dots, b_m(x)$, each of which is an indicator of one of the possible categorical values:
$$
b_i(x) = [f_j(x) = c_i]
$$

__Data preparation.__

Download [UCI Bank Marketing Dataset](https://archive.ics.uci.edu/ml/datasets/bank+marketing). This dataset contains information about bank telephone marketing, the object in it is a phone call to a potential client with an offer of some service (it is alleged that this is a short-term deposit), the target variable is the client's response (did he/she agree to open a deposit?). Characteristics of the client (education, marriage, etc.), call data and various economic indicators are used as an indicative description - more detailed information is presented in the `bank-additional-names.txt` file.

In [4]:
#!wget https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip
#!unzip bank-additional.zip
df = pd.read_csv('bank-additional/bank-additional-full.csv', sep=';')

In [5]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


__Task 3.__ Divide the sample into training and testing in a ratio of 3:1. Fix `random_state=777`, also use `stratify`. One of the columns in the data does not carry information for solving the problem - which one? Drop it.

In [10]:
#your code here

Encode categorical features using `OrdinalEncoder`. Calculate the quality (in this task we will work with `AUC-PR`) when applying logistic regression. Measure the time it took to train the model (taking into account feature encoding).

__Question__: why in this task we chose `AUC-PR` as a metric, and not, for example, `AUC-ROC`?

__Your reply__:

In [11]:
#your code here

__Task 4.__ Encode all categorical features using one-hot coding. Train logistic regression and see how the quality of the model has changed (from what it was before encoding). Measure the time required to encode categorical features and train the model.

In [12]:
#your code here

As it could be replaced, one-hot encoding can greatly increase the number of features in the dataset, which affects memory, especially if some feature has a large number of values. This problem is solved by another way of encoding categorical features - counters. The main idea is that it is not the categories themselves that are important to us, but the values ​​of the target variable that the objects of this category have. We will replace each categorical feature with the average value of the target variable for all objects of the same category:
$$
g_j(x, X) = \frac{\sum_{i=1}^{l} [f_j(x) = f_j(x_i)][y_i = +1]}{\sum_{i=1}^{l } [f_j(x) = f_j(x_i)]}
$$

__Task 5.__ Encode categorical variables with counters (exactly as described above without any tricks). Train logistic regression and look at the quality of the model on the test set. Compare training time with previous experiments (taking into account feature encoding). Did you notice something interesting?


__Bonus__: Here and below, implement the calculation of counters using a transformer (inherit from the `BaseEstimator, TransformerMixin` classes from `sklearn.base`). Please note that all calculations must be vectorized, the transformer must not modify the inplace sample passed to it, and all necessary statistics must be calculated only on the training sample in the `fit` method. Your transformer should take a list of categorical features during initialization and change only them.

In [13]:
#your code here

Note that such features are classifiers in themselves and, learning from them, we allow the target variable to “leak” into features. This leads to overfitting, so it is necessary to count such features in such a way that when calculating for a particular object, its target label is not used. This can be done in the following ways:
1. Calculate the counter value for all objects located higher in the dataset (for example, if we have a selection sorted by time).
2. Calculate by folds, that is, divide the sample into a certain number of parts and calculate the value of features for all folds except the current one (as is done in cross-validation).
3. Introduction of some noise into the calculated features.

__Task 6.__ Implement correct calculation of counters in the simplest way - adding noise to the values ​​(try to find a balance between fighting overfitting and keeping the features useful). Train logistic regression again, evaluate the quality. Draw your own conclusions.

In [71]:
#your code here

__(Bonus part)__ Calculate the correct counters in the first or second way described above (don't forget to add noise).

In [72]:
#your code here

Now answer the following question: what happens if a certain category occurs in the sample only a few times? For this reason, counters are smoothed. For example, in practice, good results are shown by using smoothing by means of the entire sample:
$$
g_j(x, X) = \frac{\sum_{i=1}^{\ell} [f_j(x) = f_j(x_i)][y_i = +1] + C \times global\_mean}{\sum_ {i=1}^{\ell} [f_j(x) = f_j(x_i)] + C}
$$
where $global\_mean$ is the proportion of objects of a positive class in the sample, $C$ is a parameter that determines the degree of smoothing (for example, you can use 10 or choose your own for each feature). The main idea is that we "dilute" the average for some category with the global average. And the less, the more objects of this category are found in the sample.

However, for smoothing, instead of the average value of the target variable, you can use any other value from 0 to 1 (this parameter is sometimes called $prior$). You can make several features with different parameter values. In practice, even negative values can be useful in binary classification problems!

__Task 7.__ Add the smoothing described above and repeat the experiments..

In [87]:
#your code here

__Task 8.__ The data contains the feature "client's age". Now we interpret it as a numerical one, which in the general case for a linear model may be an incorrect hypothesis. However, this feature has quite a few unique values (how many?), so applying one-hot encoding to it may be redundant. Try to encode age with counters. Has it gotten better?

In [1]:
#your code here

You can also go in the opposite direction. We have signs "month and day of the week" for the call. Try to interpret them as numbers (month 0 to 12, days of the week 0 to 4). Did it get better this time?

In [76]:
#your code here

__Note.__ Complicating methods for calculating counters does not guarantee better model results. Especially considering that logistic regression is not such a complex model to retrain. So you didn't have to get better and better results at each step (but you should have got the necessary results).

As you should have noticed, counters are a good alternative to one-hot coding. Write what pros and cons of using counters compared to one-hot coding you noticed.

__Answer:__

# Part 3. Feature Selection (2points)

Feature selection is an important part of the model building process. In practice, many features have little effect on the model (and their calculation takes time) or even negatively affect the quality of the model. Let's try several feature selection approaches, evaluate how they affect the quality of the model and how long they take.

Let's turn to the same dataset about bank telephone marketing.

In [82]:
df = pd.read_csv('bank-additional/bank-additional-full.csv', sep=';')

X = df.drop(columns=['duration', 'y'])
y = (df.y == 'yes')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=777, stratify=y)

As you remember, there are many categorical features in the data (for now, let's interpret age as a number). Let's encode them using one-hot encoding. The original columns with categorical features can be deleted. How many signs did we get?

In [14]:
#your code here

We will use logistic regression as the main model, and `AUC-PR` as the target metric. Train the model and calculate the quality on the test set. Let's remember the resulting value.

In [15]:
#your code here

### Built-in methods

Let's say we want to keep only the top 40 features. Let's try to do this in several ways.

Let's start with feature selection using a linear model. As you know, the weights of a linear model mean the contribution of each feature to the prediction of the model, which means that the modulus of this contribution can be interpreted as the importance of the features. Such a selection method is called built-in or embedded methods, since it is embedded in the features of the model.

__Task 8.__ Leave 40 features with the largest modulus of the corresponding parameter of the linear model. Retrain the model and evaluate its quality. Measure the speed of such feature selection.

Has the quality changed? How?

In [85]:
#your code here

Now let's think about what we didn't take into account. We assumed that the features contribute uniformly, but did not take into account their scale. If we multiply one of the features by 100 times, then without regularization, its weight will decrease by the same 100 times. And based on this, we select signs! Let's first scale the features in one of the ways, and only then we will remove the features.

By the way, in this case, you need to recalculate the quality on all features (do it below). If you did feature normalization at the very beginning, then try to select features on unscaled data.

What happened?

In [86]:
#your code here

A quick question: one-hot coding returns single indicator features to us. Try to scale them as well as regular numeric ones, and again select 40 main features by contribution. Has their list changed? Will the quality change?

In [88]:
#your code here

### Filtering methods


Let's select features smarter, namely through the calculation of some function for each feature. Based on the values ​​of this function, we will leave the most important features. The methods of this family are called filter methods.

As such a function, we will consider t-statistics:

$$t(j) = \frac{|\mu_+ - \mu_-|}{\sqrt{\frac{n_+ \sigma^2_+ + n_- \sigma^2_-}{n_+ + n_-} }},$$

where $\mu$, $\sigma$, $n$ are the mean, standard deviation, and number of objects of each class, respectively.

__Task 9.__ Leave 40 features with the highest $t$ value and measure the quality. Don't forget to measure the feature selection rate in this case.

In [89]:
#your code here

### Wrapper methods

__Task 10 (1 point).__

The third of the methods we are considering works as follows: we exclude one of the features in turn and see how this affects the quality. We remove the features in such a greedy way until some condition is met (the number of features or deterioration in quality).

Note that it is impossible to evaluate the quality of the test sample, otherwise we can overfit, as, for example, when tuning hyperparameters. Divide the training sample into 2 parts, train the model without one of the features on one of them, evaluate the quality on the second part. The initial test sample should only be used for the final quality assessment.

Again, leave only 40 features and evaluate the quality on the test sample. How long did this feature selection take?

In [90]:
#your code here

It is worth noting that using this method, you can go in the opposite direction. Try to _add_ one of the most useful features to the sample until there are 40 pieces. Is there a threshold at which adding the following features will only worsen the quality of the model?

In [91]:
#your code here

Let's sum up the feature selection. List the advantages and disadvantages of each method. Which method led to the best quality?

**Answer:**

# Part 4. Evaluation of the economic effect of the model (2points)

In this part, we will deal with what you are most likely to need in a real job (besides rewriting `json`, of course). Namely: we will collect several specialized quality metrics, try to tune the model to maximize _profit_ and estimate how much we can actually earn from this. Of course, a lot of simplifying assumptions will be made here, but first things first.

__Task 11.__ Let's say your call center employees receive $1 per call to a customer. If the client agrees to the proposed conditions, he will bring $10 to the bank.

If you listened to everything on the economy, then we recall that revenue is how much money customers brought us, and profit is revenue minus salary costs and so on.

Load the telemarketing data from the previous task block. In this part, you do not need to divide the sample - we will use cross-validation. Use 5 folds, do `shuffle=True, random_state=500`. By cross-validation, you will get 5 options for training and test sets. Train logistic regression on each training set (use one-hot for categorical features, leave hyperparameters at their defaults) and make predictions for the corresponding test sets. Let's say your employees decide to call all the positive forecasts. Calculate the revenue on all test samples and average. How much money do you earn on average? Also calculate the standard deviation.

How much of the money you earn will have to give to your call center operators?

In [16]:
#your code here

Let's add some randomness. Now let the client who agrees to the terms bring not 10 dollars, but a random value uniformly distributed in the interval $[0;20)$. Do all the same steps. To simulate a real situation, **DO NOT** fix `random_seed` when calculating customer revenue. What happened?

In [17]:
#your code here

Adjust the cross-validation coefficient of the regularization of the model to maximize profit (consider as a random variable above). Did you get any win? At what regularization coefficient and profit is maximum? Plot expected profit versus odds, also specify standard deviations (`plt.errorbar` will help you).

In [18]:
#your code here

Try to run iteration several times. Is there the same "best" coefficient every time? Is there any pattern? What conclusions can you draw from this?

In [19]:
#your code here

__Task 12.__ We have already described an approximate economic model of your task above. What do you think is more important for your business - good precision or recall models? Why?

__Answer:__


Recall that in fact, logistic regression predicts for us the probabilities of a positive class for an object. Perhaps by adjusting the binarization threshold for these probabilities, we can get some kind of gain? Test your reasoning above by adjusting the binarization threshold on cross-validation to maximize profits. Use a grid from 0 to 1 with a step of 0.01. Recall that lowering the threshold gives us higher recall and lower precision, and vice versa.

In [20]:
#your code here

Build a graph of the dependence of profit on the binarization threshold, also mark the standard deviations on it. Select the best threshold.

Question: Do you notice any pattern? For the correct answer to this question, try to run several times and think about why the threshold is obtained in a particular area?

In [21]:
#your code here

What are the values of precision and recall at the selected threshold? Evaluate by cross-validation. Also calculate the standard deviation.

In [22]:
#your code here

As you probably already figured out, your model tends to have a higher recall. Try to evaluate the quality of the model using `PR-AUC` in the $\geq$ 0.5 recall zone. Do it this way - select only those thresholds that achieve the desired recall, then interpret the unit square clipping rectangle as a new unit square and calculate the area under the clipping curve.

In [23]:
#your code here

__Bonus:__ what did you eat for breakfast on the day of this homework? ))

__Answer:__